# 1. LSTM BASED SEQUENCE PREDICTION SYSTEM WITH DEPLOYMENT

## 2. Group Details : 
### Darshan Bhabad - 202301040169
### Mitesh Chaudhari - 202301040106
### Krishna Tolani - 2023010400

## GITHUB REPO : (https://github.com/DarshanBhabad/LSTM-Based-Sequence-Prediction-System-with-Deployment)
## KAGGLE NOTEBOOK :(https://www.kaggle.com/code/bboyattitude/lstm-based-sequence-prediction-system)

## 3. Dataset Details:
**1.Dataset Name / API:** Wikipedia API (MediaWiki)

**2.Source Link:** Wikipedia 

**Python WrapperDescription:** This dataset consists of full-text articles fetched dynamically from Wikipedia regarding "Artificial Intelligence" and "Machine Learning" to provide a technical corpus for sequence prediction.
Preprocessing Steps: 
1.  Lowercasing text.
2.  Removing Wikipedia-specific headers (e.g., "See Also", "References").
3.  Regex cleaning to remove special characters and citations (e.g., [1])
4.  Tokenization and $n$-gram sequence creation.

# Step 4.1: Install & Import Libraries

In [1]:
!pip install wikipedia-api  # Specific library for Wikipedia fetching

import numpy as np
import tensorflow as tf
import wikipediaapi
import re
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print("Environment Ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


2026-04-18 14:50:23.560834: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776523823.801998      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776523823.874100      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776523824.413206      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776523824.413251      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776523824.413254      55 computation_placer.cc:177] computation placer alr

Environment Ready.


# Step 4.2: Dataset Collection via API

In [2]:
# Initialize Wikipedia API
# Use a custom user_agent as per Wikipedia policy
wiki = wikipediaapi.Wikipedia(
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI,
    user_agent="CollegeProject/1.0 (contact: yourname@example.com)"
)

# Fetching content from multiple related pages
topics = ["Artificial intelligence", "Machine learning", "Deep learning", "Neural network"]
raw_corpus = ""

for topic in topics:
    page = wiki.page(topic)
    if page.exists():
        raw_corpus += page.text + " "

print(f"Total characters fetched: {len(raw_corpus)}")

Total characters fetched: 203956


# Step 4.3: Data Preprocessing

In [3]:
def clean_text(text):
    text = text.lower()
    # Remove citations like [1], [22]
    text = re.sub(r'\[[0-9]*\]', '', text)
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = " ".join(text.split())
    return text

cleaned_corpus = clean_text(raw_corpus)

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([cleaned_corpus])
total_words = len(tokenizer.word_index) + 1

# Generating Sequences
token_list = tokenizer.texts_to_sequences([cleaned_corpus])[0]
input_sequences = []
for i in range(1, len(token_list)):
    # Create a sliding window sequence (e.g., 10 words)
    n_gram_sequence = token_list[max(0, i-10):i+1]
    input_sequences.append(n_gram_sequence)

# Padding
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# X and Y split
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Step 4.4: Model Development

In [4]:
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len-1),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=50, batch_size=64) # Adjust epochs based on time

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1776523852.663965      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776523852.670045      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/50


I0000 00:00:1776523859.354580     133 cuda_dnn.cc:529] Loaded cuDNN version 91002


465/465 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.0407 - loss: 7.3938
Epoch 2/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.0503 - loss: 6.7143
Epoch 3/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.0643 - loss: 6.5639
Epoch 4/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.0748 - loss: 6.4119
Epoch 5/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.0867 - loss: 6.2709
Epoch 6/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.0997 - loss: 6.1100
Epoch 7/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1070 - loss: 6.0125
Epoch 8/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1232 - loss: 5.8964
Epoch 9/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1309 - loss: 5.7827
Epoch 10/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1346 - loss: 5.6538
Epoch 11/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1376 - loss: 5.5669
Epoch 12/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy

# Step 4.5: Saving Trained Model

In [5]:
# Exporting files for the FastAPI Deployment
model.save("lstm_text_model.h5")

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('config.pkl', 'wb') as f:
    pickle.dump({'max_seq_len': max_sequence_len}, f)

print("All files exported for deployment Part 2.")

All files exported for deployment Part 2.
